In [47]:
import os
import pandas as pd
import uuid

In [58]:
df_paths = [
    'token_data_0_20000.tsv',
    'token_data_20000_40000.tsv',
]
df = pd.DataFrame()
for df_path in df_paths: 
    temp = pd.read_csv(df_path, sep='\t')
    print(df_path, len(temp), temp.columns)
    df = pd.concat([df, temp], axis=0, ignore_index=True)
df.drop_duplicates(subset=['token_hash'], inplace=True)
len(df), df.columns

token_data_0_20000.tsv 11877 Index(['token_hash', 'word', 'lemma', 'pos', 'xpos', 'deprel', 'count',
       'sentences', 'group_hash', 'group_count', 'group_pct', 'word_pct'],
      dtype='object')
token_data_20000_40000.tsv 10701 Index(['token_hash', 'word', 'lemma', 'pos', 'xpos', 'deprel', 'count',
       'sentences', 'group_hash', 'group_count', 'group_pct', 'word_pct'],
      dtype='object')


(16424,
 Index(['token_hash', 'word', 'lemma', 'pos', 'xpos', 'deprel', 'count',
        'sentences', 'group_hash', 'group_count', 'group_pct', 'word_pct'],
       dtype='object'))

In [ ]:
def get_group_hash(lemma, pos):
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{lemma}-{pos}"))
df['group_hash'] = df.apply(lambda row: get_group_hash(row['lemma'], row['pos']), axis=1)

In [60]:
group_counts = {}
for group in df['group_hash'].unique():
    mask = df['group_hash'] == group
    group_df = df[mask]
    df.loc[mask, 'group_count'] = group_df['count'].sum()
    group_counts[group] = group_df['count'].sum()

In [61]:
df['group_pct'] = df['group_count'] / df['count'].sum()
df['word_pct'] = df['count'] / df['count'].sum()

In [62]:
df.sort_values(by=['group_pct', 'word_pct'], ascending=False, inplace=True)

In [63]:
print(len(df))
df = df[df['pos'] != 'PROPN']
print(len(df))

16424
15130


In [66]:
df.to_csv('token_data.tsv', sep='\t', index=False)

In [65]:
df.head(20)

,token_hash,word,lemma,pos,xpos,deprel,count,sentences,group_hash,group_count,group_pct,word_pct
0,ca056673-0fec-51df-aa78-91d137c6be22,il,il,DET,RD,det,3186,"['64752c4f-7dce-5231-b89b-fc9d435c0d70', 'd1ce...",e5de14ca-b785-57e7-ae0b-4d8d72b47700,9129.0,0.069085,0.024110
1,70af3935-65f5-5270-9006-6d8a9ad933b5,la,il,DET,RD,det,2973,"['d1ce4849-0042-58cd-be07-28b0a685aa4a', '1f0c...",e5de14ca-b785-57e7-ae0b-4d8d72b47700,9129.0,0.069085,0.022499
2,79de0ccb-2172-5399-b99f-ade09a88c14d,l',il,DET,RD,det,1054,"['8f00a5a2-fd20-5477-baee-247f88ffaeed', 'fb53...",e5de14ca-b785-57e7-ae0b-4d8d72b47700,9129.0,0.069085,0.007976
3,27fbf1e4-d46d-5226-b72f-e83f4d1ba0d9,i,il,DET,RD,det,839,"['f8d997c6-a967-5859-803c-af956680d7ef', '2f6e...",e5de14ca-b785-57e7-ae0b-4d8d72b47700,9129.0,0.069085,0.006349
4,0232d8ec-3c64-5fe6-a912-c68f252eba93,le,il,DET,RD,det,719,"['b82d679f-1ba1-5718-b4fd-24f8a3b72fa8', '718f...",e5de14ca-b785-57e7-ae0b-4d8d72b47700,9129.0,0.069085,0.005441
5,9ac39d22-f599-52d2-9387-74c8ded2b229,gli,il,DET,RD,det,259,"['dc7a354b-18bb-5586-ace4-95801df98c47', 'e838...",e5de14ca-b785-57e7-ae0b-4d8d72b47700,9129.0,0.069085,0.001960
6,569615ad-c38c-555e-b2a7-af9245f5a17b,lo,il,DET,RD,det,99,"['10679083-51fe-5074-a2e0-49babe4fbb76', 'a416...",e5de14ca-b785-57e7-ae0b-4d8d72b47700,9129.0,0.069085,0.000749
7,d05e0172-8c49-5787-87f1-9016a42d14f2,è,essere,AUX,VA,cop,4133,"['64752c4f-7dce-5231-b89b-fc9d435c0d70', '3a0b...",3a4ed285-f371-52af-9a51-996aedf5a772,7770.0,0.058800,0.031277
8,c72de050-9a00-5ae2-aea5-4c4bfe357355,sono,essere,AUX,VA,cop,1379,"['560b68e5-dd8f-5730-9f73-3e6b2633e1d3', 'ed1c...",3a4ed285-f371-52af-9a51-996aedf5a772,7770.0,0.058800,0.010436
9,af27c950-5479-51ba-9aa2-c12080378bac,era,essere,AUX,VA,cop,431,"['5453a566-bf70-51f9-884a-6cd98c1342ee', '1ff8...",3a4ed285-f371-52af-9a51-996aedf5a772,7770.0,0.058800,0.003262


In [55]:
df[df['lemma'] == 'essere'][['word', 'pos', 'lemma']]

,word,pos,lemma
7,è,AUX,essere
11884,è,AUX,essere
8,sono,AUX,essere
11885,sono,AUX,essere
9,era,AUX,essere
...,...,...,...
16446,essere,NOUN,essere
5964,esseri,NOUN,essere
16505,esseri,NOUN,essere
6013,essere,NOUN,essere


In [56]:
word = 'gli'
df[df['word'] == 'gli'][['word', 'pos', 'lemma']]

,word,pos,lemma
5,gli,DET,il
11882,gli,DET,il
1100,gli,PRON,gli
13090,gli,PRON,gli
